# 02 - SQL Validation

## Objective

In this notebook, I validate the raw Telco customer data after importing it into PostgreSQL.

The goal here is not to clean the full dataset or build features yet. I want to check whether the data looks consistent after moving from CSV into a database.

The main questions are:

- Did all rows import correctly?
- Does the SQL row count match the pandas audit?
- Are missing values represented the same way in SQL?
- Is the target column imported correctly?
- Are there any differences between what pandas showed and what SQL shows?

This step helps me confirm that the raw data layer is reliable before moving into deeper analysis and modeling.

### Why SQL-Based Validation?

I wanted to use SQL in this project not just to show that the data can be stored in a database, but also to validate the raw data from another angle.

In many real projects, the data does not live only in a CSV file. It usually sits in a database, and analysts need to check the data directly from there.

For this reason, I used PostgreSQL to:

- confirm the row count after import
- inspect the imported table
- check how missing values are represented
- validate the target column distribution
- compare SQL results with the earlier pandas audit

I also used environment variables for the database password so that credentials are not written directly inside the notebook.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", None)

In [2]:
DB_USER = "postgres"
DB_PASSWORD = os.getenv("PGPASSWORD")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "telco_churn_db"

In [3]:
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

engine = create_engine(connection_url)

In [4]:
# %pip install psycopg2-binary sqlalchemy

In [5]:
query = "SELECT current_database();"
pd.read_sql(query, engine)

,current_database
0,telco_churn_db


## Validate Row Count

The first thing I checked was whether PostgreSQL contains the same number of rows as the original dataset.

This is a simple check, but it is important. If the row count does not match at this stage, then any later analysis or modeling result may be based on an incomplete import.

In [6]:
query = """
SELECT COUNT(*) AS total_rows
FROM telco_customers;
"""
pd.read_sql(query, engine)

,total_rows
0,7043


## Preview Imported Rows

After checking the row count, I previewed a few records from the SQL table.

This helps confirm that the table was imported in the expected structure and that the columns are readable from PostgreSQL.

In [7]:
query = """
SELECT *
FROM telco_customers
LIMIT 5;
"""
pd.read_sql(query, engine)

,customer_id,gender,age,under_30,senior_citizen,married,dependents,number_of_dependents,country,state,city,zip_code,latitude,longitude,population,quarter,referred_a_friend,number_of_referrals,tenure_in_months,offer,phone_service,avg_monthly_long_distance_charges,multiple_lines,internet_service,internet_type,avg_monthly_gb_download,online_security,online_backup,device_protection_plan,premium_tech_support,streaming_tv,streaming_movies,streaming_music,unlimited_data,contract,paperless_billing,payment_method,monthly_charge,total_charges,total_refunds,total_extra_data_charges,total_long_distance_charges,total_revenue,satisfaction_score,customer_status,churn_label,churn_score,cltv,churn_category,churn_reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,Los Angeles,90022,34.023810,-118.156582,68701,Q3,No,0,1,None,No,0.00,No,Yes,DSL,8,No,No,Yes,No,No,Yes,No,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,Los Angeles,90063,34.044271,-118.185237,55668,Q3,Yes,1,8,Offer E,Yes,48.85,Yes,Yes,Fiber Optic,17,No,Yes,No,No,No,No,No,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,Los Angeles,90065,34.108833,-118.229715,47534,Q3,No,0,18,Offer D,Yes,11.33,Yes,Yes,Fiber Optic,52,No,No,No,No,Yes,Yes,Yes,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,Inglewood,90303,33.936291,-118.332639,27778,Q3,Yes,1,25,Offer C,Yes,19.76,No,Yes,Fiber Optic,12,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,Whittier,90602,33.972119,-118.020188,26265,Q3,Yes,1,37,Offer C,Yes,6.33,Yes,Yes,Fiber Optic,14,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


## Investigate Missing-Like Categorical Values

### Key Finding: Missing values are not represented in one single way

During SQL validation, I noticed an important detail about missing values.

Some columns do not use SQL `NULL`. Instead, they contain the literal text `"None"`.

For example:

- `offer`
- `internet_type`

But other churn-related columns use proper SQL `NULL` values:

- `churn_category`
- `churn_reason`

This means I cannot clean all missing values with one simple rule.

If I only check for SQL `NULL`, I will miss the `"None"` strings.  
If I only replace `"None"`, I will miss real SQL `NULL` values.

Because of this, missing-value handling needs to be column-specific in the preprocessing notebook.

In [8]:
query = """
SELECT offer, COUNT(*) AS cnt
FROM telco_customers
GROUP BY offer
ORDER BY cnt DESC;
"""
pd.read_sql(query, engine)

,offer,cnt
0,None,3877
1,Offer B,824
2,Offer E,805
3,Offer D,602
4,Offer A,520
5,Offer C,415


### Check all columns that showed missingness in the pandas audit

Instead of checking only one column, I checked all columns that showed missingness during the pandas audit.

The goal was to understand whether those missing values became SQL `NULL`, literal `"None"` strings, empty strings, or something else after import.

In [9]:
missing_like_cols = ["offer", "internet_type", "churn_category", "churn_reason"]

results = []

for col in missing_like_cols:
    query = f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN {col} = 'None' THEN 1 ELSE 0 END) AS none_count
    FROM telco_customers;
    """
    result = pd.read_sql(query, engine)
    results.append(result)

missing_like_summary = pd.concat(results, ignore_index=True)
missing_like_summary

,column_name,total_rows,none_count
0,offer,7043,3877
1,internet_type,7043,1526
2,churn_category,7043,0
3,churn_reason,7043,0


### Investigate columns where `"None"` was not found

The previous query showed that `churn_category` and `churn_reason` do not contain literal `"None"` values.

Since these columns still had missing values in the pandas audit, I checked whether the missing values were stored as SQL `NULL` or empty strings.

In [10]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(churn_category) AS non_null_churn_category,
    SUM(CASE WHEN churn_category IS NULL THEN 1 ELSE 0 END) AS sql_null_churn_category,
    SUM(CASE WHEN churn_category = '' THEN 1 ELSE 0 END) AS empty_string_churn_category,
    COUNT(churn_reason) AS non_null_churn_reason,
    SUM(CASE WHEN churn_reason IS NULL THEN 1 ELSE 0 END) AS sql_null_churn_reason,
    SUM(CASE WHEN churn_reason = '' THEN 1 ELSE 0 END) AS empty_string_churn_reason
FROM telco_customers;
"""
pd.read_sql(query, engine)

,total_rows,non_null_churn_category,sql_null_churn_category,empty_string_churn_category,non_null_churn_reason,sql_null_churn_reason,empty_string_churn_reason
0,7043,1869,5174,0,1869,5174,0


### Interpretation

This check confirmed that missing values are represented differently across columns.

Main observations:

- `churn_category` contains 5,174 SQL `NULL` values.
- `churn_reason` also contains 5,174 SQL `NULL` values.
- There are no empty strings in these two columns.
- This is different from `offer` and `internet_type`, where missing-like values appeared as literal `"None"` text.

The main takeaway is that missing-value handling should not be automatic or careless.

In the preprocessing stage, I will need to normalize these values based on how each column stores missing information.

## Validate Target Candidate

After checking missing-value representation, I validated the target candidate: `churn_label`.

This check confirms whether the target column was imported correctly and whether the class distribution still matches the earlier pandas audit.

In [11]:
query = """
SELECT churn_label, COUNT(*) AS cnt
FROM telco_customers
GROUP BY churn_label
ORDER BY cnt DESC;
"""
pd.read_sql(query, engine)

,churn_label,cnt
0,No,5174
1,Yes,1869


### Interpretation

The `churn_label` column was imported correctly as a binary outcome field.

The class distribution matches the earlier audit:

- `No` = 5,174
- `Yes` = 1,869

This confirms that `churn_label` can be used as the target variable in the modeling stage.

It also shows that the dataset is imbalanced, but not extremely imbalanced. Because of this, I will evaluate models using metrics beyond accuracy, especially precision, recall, F1-score, and ROC-AUC.

## Validation Summary

This notebook helped me confirm that the PostgreSQL import was successful and that the raw data can be trusted for the next steps.

Main conclusions:

- The SQL row count matches the original dataset: 7,043 rows.
- The target distribution matches the pandas audit: 5,174 non-churn customers and 1,869 churn customers.
- `churn_label` is a valid binary target candidate.
- Missing values are not represented consistently across all columns.
- `offer` and `internet_type` contain literal `"None"` strings.
- `churn_category` and `churn_reason` contain SQL `NULL` values.

The most important lesson from this step is that data cleaning should be based on actual validation, not assumptions.

Because of this, I will handle missing-like values carefully in Python preprocessing instead of applying one global cleaning rule to every column.